In [ ]:
import os
from dotenv import load_dotenv
from pyjstat import pyjstat
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

import pymongo
import psycopg2
import pandas as pd
import requests

In [ ]:
# download data 
api_url = "https://ws.cso.ie/public/api.restful/PxStat.Data.Cube_API.ReadDataset/CJQ06/JSON-stat/1.0/en"
response = requests.get(api_url)

# extract the JSON-STAT file and store it into data frame of pandas
dataset = pyjstat.Dataset.read(response.text)
df = dataset.write('dataframe')

# load env variable to notebook
load_dotenv(override=True)
# call and store mongoDB protected detail
mongo_url = os.getenv("MONGO_URI")
if mongo_url:
    print(f"Load URI successfully: {len(mongo_url)} ")
else:
    print("Warning: Can not find MONGO_URI in file .env!")

# connect to MongoDB
client = pymongo.MongoClient(mongo_url)
db = client['IE_Crime_Project']
collection = db['raw_crime_data']

# convert df to dictionary to pull into MongoDB
data_dict = df.to_dict(orient='records')
collection.drop() 
collection.insert_many(data_dict)

In [ ]:
load_dotenv(override=True)
# call and store mongoDB protected detail
mongo_url = os.getenv("MONGO_URI")
client = pymongo.MongoClient(mongo_url)
db = client['IE_Crime_Project']
collection = db['raw_crime_data']
# Fetch data from the MongoDB
raw_data = list(collection.find())
# Convert JSON to dataframe
df = pd.DataFrame(raw_data)

In [ ]:
df.columns

In [ ]:
df.head(5)

In [ ]:

# Drop the id column automatically generated by MongoDB
if '_id' in df.columns:
    df = df.drop(columns=['_id'])

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
null_rows = df[df['value'].isnull()]
print(len(null_rows))
print(null_rows)

In [ ]:
null_rows.nunique()

In [ ]:
for i in null_rows.columns:
    print(null_rows[i].unique())
    print("-" * 30)

In [ ]:
# fill the null data point in the value column by 0 
df['value'] = df['value'].fillna(0)

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe(include=['object'])

In [ ]:
non_numeric_df = df.select_dtypes(exclude=['number'])
for i in non_numeric_df:
    print(df[i].unique())
    print(len(df[i].unique()))
    print("-" * 30)

In [ ]:
non_integer_rows = df[df['value'] % 1 != 0]

print(non_integer_rows)

In [ ]:
df['year'] = df['Quarter'].str[:4].astype(int)

df['quarter_number'] = df['Quarter'].str[-1].astype(int)

# Check
print(df[['Quarter', 'year', 'quarter_number']].head())
print(df[['Quarter', 'year', 'quarter_number']].tail())

In [ ]:
print(df['year'].unique())
print(df['year'].dtype)
print("-"*30)
print(df['quarter_number'].unique())
print(df['quarter_number'].dtype)

In [ ]:
# extract county from garda division
df['County'] = df['Garda Division'].str.replace(' Garda Division', '', case=False).str.strip()

# Check result
print(df[['Garda Division', 'County']].drop_duplicates().head())
print(df[['Garda Division', 'County']].drop_duplicates().tail())

In [ ]:
print(len(df['Garda Division'].unique()))
print("-"*30)
print(df['County'].unique())
print(len(df['County'].unique()))
print(len(df['County'].dtype))

In [ ]:
# exclude columns
exclude_cols = ['Quarter', 'Garda Division', 'Statistic']

# Filter to get the columns which are not in the excluding list
cols_to_plot = [col for col in df.columns if col not in exclude_cols]

# 3. Thiết lập kích thước và phong cách
# sns.set(style="whitegrid")

# Create Subplots
n_plot = len(cols_to_plot)
nrows = (n_plot + 1) // 2
ncols = min(n_plot, 2)

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(15, 7 * nrows))
axes = axes.flatten()

for i, col in enumerate(cols_to_plot):
    if df[col].dtype == 'object':
        # Ibject: count plot
        top_values = df[col].value_counts().head(10).index
        labels = [(str(x)[:25] + '...') if len(str(x)) > 25 else str(x) for x in top_values]
        sns.countplot(data=df[df[col].isin(top_values)], y=col, ax=axes[i], order=top_values)
        axes[i].set_yticklabels(labels)
        axes[i].set_title(f'Distribution of {col}')
    else:
        # Numeric: Histogram plot
        sns.histplot(df[col].dropna(), kde=True, ax=axes[i])
        axes[i].set_title(f'Histogram of {col}')

# Delete redundant subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
df['value'].max()

In [ ]:
df[df['value'] > 4000]

In [ ]:
df.head(10)

In [ ]:
df.drop(columns = ["Quarter","Garda Division"])

In [ ]:
df['Group_County'] = df['County']

df.loc[df['County'].str.startswith('Cork', na=False), 'Group_County'] = 'Cork'
df.loc[df['County'].str.startswith('D.M.R.', na=False), 'Group_County'] = 'Dublin'

# Chạy bước 1 và 2 xong, hãy dùng lệnh này để kiểm tra:
check_cork = df[df['County'].str.startswith('Cork', na=False)][['County', 'Group_County']].drop_duplicates()
check_dmr = df[df['County'].str.startswith('D.M.R.', na=False)][['County', 'Group_County']].drop_duplicates()

print("Group Cork")
print(check_cork)
print("\nGroup Dublin")
print(check_dmr)
df['Group_County'].unique()

In [ ]:
group_cols = ['Group_County', 'Statistic', 'Type of Offence', 'year', 'quarter_number']
df_summed = df.groupby(group_cols, as_index=False)['value'].sum()
df_summed = df_summed.rename(columns={'Group_County': 'County'})
print(df_summed[df_summed['County'] == "Dublin"])

In [ ]:
df_summed.describe()
df_summed.info()
df_summed.columns
df_summed.head(5)

In [ ]:
df_summed = df_summed.rename(columns={
    'quarter_number': 'quarter'
})
df_summed.columns = [c.replace(' ', '_').lower() for c in df_summed.columns]

In [ ]:
df_summed.columns


In [ ]:
group_cols = ['county', 'statistic', 'type_of_offence', 'year']
df_summed['sum_by_group'] = df_summed.groupby(group_cols)['value'].transform('sum')
group_cols_2 = ['county', 'statistic', 'type_of_offence', 'year','sum_by_group']
df_summed = df_summed.groupby(group_cols_2, as_index=False)['value'].sum()
df_summed.head(10)

In [ ]:
is_all_equal = (df_summed["sum_by_group"] == df_summed["value"]).all()
print(is_all_equal)

In [ ]:
df_summed.nunique()

In [ ]:
df_summed = df_summed.drop(columns=["statistic", "sum_by_group"])

In [ ]:
df_summed.head()

In [ ]:
sns.set(style="whitegrid")

cols_to_plot = ['county', 'year', 'type_of_offence']

fig, axes = plt.subplots(nrows=len(cols_to_plot), ncols=1, figsize=(12, 6 * len(cols_to_plot)))

for i, col in enumerate(cols_to_plot):
    sns.barplot(
        data=df_summed, 
        x=col, 
        y='value', 
        estimator=sum, 
        errorbar=None, 
        ax=axes[i],
        palette='viridis'
    )
    
    axes[i].set_title(f'Sum Value by {col}', fontsize=15)
    axes[i].set_ylabel('Sum Value')
    axes[i].set_xlabel(col)
    
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
print(df_summed.dtypes)

In [ ]:
df_summed["value"] = df_summed["value"].astype(int)

In [ ]:
print(df_summed.dtypes)

In [ ]:
df_summed["county"].unique()

In [ ]:
data = {
    "NUTS2_Code": ["IE04"] * 6 + ["IE05"] * 8 + ["IE06"] * 7,
    
    "NUTS2_Name": (
        ["Northern & Western"] * 6 + 
        ["Southern"] * 8 + 
        ["Eastern & Midland"] * 7
    ),
    
    "NUTS3_Code": (
        ["IE041"] * 3 + ["IE042"] * 3 +        # Northern & Western
        ["IE051"] * 3 + ["IE052"] * 3 + ["IE053"] * 2 +  # Southern
        ["IE061"] * 1 + ["IE062"] * 4 + ["IE063"] * 2    # Eastern & Midland
    ),
    
    "NUTS3_Name": (
        ["Border"] * 3 + ["West"] * 3 + 
        ["Mid-West"] * 3 + ["South-East"] * 3 + ["South-West"] * 2 + 
        ["Dublin"] * 1 + ["Mid-East"] * 4 + ["Midlands"] * 2
    ),
    
    "county": [
        # Northern & Western (6)
        "Donegal", "Sligo/Leitrim", "Cavan/Monaghan", # Border
        "Galway", "Mayo", "Roscommon/Longford",      # West (Gán Ros/Lon vào đây)
        
        # Southern (8)
        "Clare", "Tipperary", "Limerick",             # Mid-West
        "Waterford", "Kilkenny/Carlow", "Wexford",    # South-East
        "Cork", "Kerry",                              # South-West
        
        # Eastern & Midland (7)
        "Dublin",                                     # Dublin
        "Wicklow", "Kildare", "Meath", "Louth",       # Mid-East
        "Westmeath", "Laois/Offaly"                   # Midlands
    ]
}

df_region_mapping = pd.DataFrame(data)

In [ ]:
df_region_mapping

In [ ]:
df_region_mapping.columns = [c.replace(' ', '_').lower() for c in df_region_mapping.columns]

In [ ]:
df_region_mapping.head(5)

In [ ]:
df_region_mapping.to_sql("region_mapping", engine, if_exists="replace", index=False)
print("Successfully upload dataset df_region_mapping into PostgreSQL!")

In [ ]:
sql_create_view = """
CREATE OR REPLACE VIEW crime_with_regions AS
SELECT c.*, 
    m."nuts2_name"
FROM  cleaned_crime_data c
LEFT JOIN region_mapping m ON c.county = m."county"  ;
"""

with engine.connect() as connection:
    connection.execute(text(sql_create_view))
    connection.commit() 
    print("Successdully creat view!")

In [ ]:
pg_user = os.getenv("PG_USER")
pg_pass = os.getenv("PG_PASSWORD")
pg_host = os.getenv("PG_HOST")
pg_port = os.getenv("PG_PORT", "5432")
pg_db = os.getenv("PG_DATABASE")
pg_uri = f"postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}"
engine = create_engine(pg_uri)

# Pull data into 'cleaned_crime_data'
df_summed.to_sql('cleaned_crime_data', engine, if_exists='replace', index=False)

print("Successfully upload dataset into PostgreSQL!")

In [ ]:
sql_create_view = """
DROP VIEW IF EXISTS crime_year_county;
CREATE OR REPLACE VIEW crime_year_county AS
SELECT SUM(value) as value,
    county, 
    year,
    nuts2_name
FROM crime_with_regions
GROUP BY county, year, nuts2_name  ;
"""

with engine.connect() as connection:
    try:
        connection.execute(text(sql_create_view))
        connection.commit()
        print("Successfully updated view: crime_year_county!")
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
df_check = pd.read_sql("SELECT * FROM crime_year_county LIMIT 5", engine)
print(df_check)

In [ ]:
sql_create_view = """
DROP VIEW IF EXISTS crime_year_region;
CREATE OR REPLACE VIEW crime_year_region AS
SELECT SUM(value) as value, 
    year,
    nuts2_name
FROM crime_with_regions
GROUP BY year, nuts2_name  ;
"""

with engine.connect() as connection:
    try:
        connection.execute(text(sql_create_view))
        connection.commit()
        print("Successfully updated view: crime_year_region!")
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
df_check = pd.read_sql("SELECT * FROM crime_year_region LIMIT 5", engine)
print(df_check)

In [ ]:
!pip install sqlalchemy 

In [ ]:
!pip install pyjstat python-dotenv pymongo psycopg2-binary requests pandas